In [0]:
-- cleaned sales with type casting, standardization, and data quality expectations
CREATE OR REFRESH STREAMING TABLE cleaned_sales (
    CONSTRAINT valid_invoice
        EXPECT (invoice_line_no IS NOT NULL)
        ON VIOLATION DROP ROW,
    CONSTRAINT valid_date
        EXPECT (sale_date IS NOT NULL AND sale_date >= '2025-01-01')
        ON VIOLATION DROP ROW,
    CONSTRAINT valid_store
        EXPECT (store_id IS NOT NULL)
        ON VIOLATION DROP ROW,
    CONSTRAINT positive_sale
        EXPECT (sale_dollars > 0)
        ON VIOLATION DROP ROW,
    CONSTRAINT positive_bottles
        EXPECT (bottles_sold > 0)
        ON VIOLATION DROP ROW,
    CONSTRAINT reasonable_amount
        EXPECT (sale_dollars < 500000)
)
AS SELECT
    invoice_line_no,
    CAST(date AS DATE)                         AS sale_date,
    CAST(store AS INT)                         AS store_id,
    UPPER(TRIM(name))                          AS store_name,
    UPPER(TRIM(address))                       AS address,
    UPPER(TRIM(city))                          AS city,
    CAST(zipcode AS STRING)                    AS zip_code,
    UPPER(TRIM(county))                        AS county,
    store_location,
    CAST(vendor_no AS INT)                     AS vendor_id,
    UPPER(TRIM(vendor_name))                   AS vendor_name,
    CAST(category AS INT)                      AS category_code,
    UPPER(TRIM(category_name))                 AS category_name,
    CAST(itemno AS INT)                        AS item_id,
    UPPER(TRIM(im_desc))                       AS item_description,
    CAST(pack AS INT)                          AS pack,
    CAST(bottle_volume_ml AS INT)              AS bottle_volume_ml,
    CAST(state_bottle_cost AS DECIMAL(10,2))   AS state_bottle_cost,
    CAST(state_bottle_retail AS DECIMAL(10,2)) AS state_bottle_retail,
    CAST(sale_bottles AS INT)                  AS bottles_sold,
    CAST(sale_dollars AS DECIMAL(12,2))        AS sale_dollars,
    CAST(sale_liters AS DECIMAL(10,4))         AS volume_sold_liters,
    CAST(sale_gallons AS DECIMAL(10,4))        AS volume_sold_gallons,
    _ingestion_ts
FROM STREAM(dev.bronze.raw_liquor_sales)